# Day 8 — Databricks SQL: DDL, DML & the Metastore

**What you will learn:**
- The metastore hierarchy: Database → Table / View
- Managed vs External tables — the most common exam trap
- `CREATE TABLE` syntax: `USING`, `OPTIONS`, `LOCATION`
- CREATE TABLE AS SELECT (CTAS)
- Persistent views (`CREATE VIEW`) vs temp views
- `spark.table()` — loading a registered table as a DataFrame
- Delta Lake DML: `INSERT INTO`, `INSERT OVERWRITE`, `UPDATE`, `DELETE`, `MERGE INTO`

**Exam relevance:** Spark SQL (20%) — DDL/DML and the metastore are directly tested.
Day 8 fills the gaps in the 16-question certification deck that Days 1-7 did not cover.

## 1. The Metastore Hierarchy

The **metastore** is a central catalogue that stores metadata about databases, tables, and views.

```
Catalog (hive_metastore or Unity Catalog)
└── Database / Schema
    ├── Tables  (physical data)
    └── Views   (stored query definitions — no physical data)
```

| Object | Stores data? | Persists across sessions? | Survives DROP? |
|---|---|---|---|
| Temp view | No | No (session-scoped) | — |
| Global temp view | No | No (app-scoped) | — |
| Persistent view | No | **Yes** | Metadata only removed |
| Managed table | **Yes** | **Yes** | **Data deleted** |
| External table | **Yes** | **Yes** | Data **survives** |

> **Exam tip:** Dropping a **managed** table deletes both the metadata AND the data files.
> Dropping an **external** table deletes only the metadata — the files remain.

In [ ]:
# Create a database to work in — keeps examples isolated
spark.sql("CREATE DATABASE IF NOT EXISTS day8_demo")
spark.sql("USE day8_demo")

# Verify current database
spark.sql("SELECT current_database()").show()

## 2. Managed vs External Tables

### Managed table
- **Omit the LOCATION clause** — Spark picks the default warehouse path
- DROP TABLE deletes the data files too

### External table
- **Include a LOCATION clause** pointing to your own path
- DROP TABLE keeps the data files — only metadata is removed

> **Exam question:** "Spark controls both data AND metadata" → CREATE TABLE with **no LOCATION** clause.

In [ ]:
# ── Managed table — no LOCATION clause ──────────────────────────────────────
spark.sql("""
    CREATE TABLE IF NOT EXISTS day8_demo.sales_managed (
        id      BIGINT,
        country STRING,
        amount  DOUBLE
    ) USING DELTA
""")

# Spark decides where to store the data (inside the warehouse directory)
spark.sql("DESCRIBE EXTENDED day8_demo.sales_managed") \
     .filter("col_name = 'Location'") \
     .show(truncate=False)

In [ ]:
# ── External table — explicitly set LOCATION ─────────────────────────────────
from pyspark.sql import Row

sample_data = [Row(id=1, country="NL", amount=1500.0),
               Row(id=2, country="DE", amount=2300.0),
               Row(id=3, country="RO", amount=900.0)]

spark.createDataFrame(sample_data) \
     .write.mode("overwrite") \
     .parquet("/tmp/day8_external_data")

spark.sql("""
    CREATE TABLE IF NOT EXISTS day8_demo.sales_external
    USING PARQUET
    LOCATION '/tmp/day8_external_data'
""")

spark.sql("SELECT * FROM day8_demo.sales_external").show()

# DROP day8_demo.sales_external  → Parquet files at /tmp/day8_external_data remain
# DROP day8_demo.sales_managed   → Delta files are permanently deleted

## 3. CREATE TABLE Syntax — USING and OPTIONS

The `USING` clause specifies the **data source format**:

```sql
CREATE TABLE table_name
USING <format>
OPTIONS (key='value', ...)
LOCATION '/path/to/data'      -- omit for managed table
```

| Format | Notes |
|---|---|
| `DELTA` | Databricks default — transactional, supports DML |
| `PARQUET` | Columnar, schema embedded in file metadata |
| `CSV` | Requires OPTIONS for header / delimiter |
| `JSON` | Schema inferred from structure |

> **Exam trap:** The keyword is `USING` — not `FROM`, `FORMAT`, or `AS`.
> `OPTIONS` sets reader properties (header, delimiter). `LOCATION` makes the table external.

In [ ]:
# CREATE TABLE over a CSV file — USING + OPTIONS + LOCATION
spark.sql("""
    CREATE TABLE IF NOT EXISTS day8_demo.flights_csv
    USING CSV
    OPTIONS (header='true', inferSchema='true', delimiter=',')
    LOCATION '/databricks-datasets/flights/departuredelays.csv'
""")

spark.sql("SELECT * FROM day8_demo.flights_csv LIMIT 5").show()

## 4. CREATE TABLE AS SELECT (CTAS)

CTAS creates **and** populates a table in a single statement.
The schema is inferred from the SELECT result — no need to define columns manually.

```sql
CREATE OR REPLACE TABLE new_table
USING DELTA
AS
SELECT col1, col2, ...
FROM source_table
WHERE condition
```

> CTAS tables are **managed** by default (no LOCATION clause).
> The resulting table is writable — supports full DML.

In [ ]:
# CTAS — create a managed Delta table from the CSV-backed table
spark.sql("""
    CREATE OR REPLACE TABLE day8_demo.flights_delta
    USING DELTA
    AS
    SELECT
        CAST(date      AS STRING) AS flight_date,
        CAST(delay     AS INT)    AS delay_minutes,
        CAST(distance  AS INT)    AS distance_miles,
        origin,
        destination
    FROM day8_demo.flights_csv
    WHERE delay IS NOT NULL
      AND distance > 0
    LIMIT 10000
""")

spark.sql("SELECT COUNT(*) AS rows FROM day8_demo.flights_delta").show()

## 5. Persistent Views — CREATE VIEW

A **persistent view** is stored in the metastore and survives across sessions.
It contains only a query definition — **no physical data is stored**.

| | Temp view | Persistent view |
|---|---|---|
| Created with | `createOrReplaceTempView()` | `CREATE VIEW` |
| Stored in metastore | No | **Yes** |
| Cross-session | No | **Yes** |
| Physical data | No | No |
| SQL prefix needed | None | `database.view_name` |

> **Exam question:** "Cross-session relational object, no physical data copy" → **persistent view**.
> Not a temp view (session-scoped). Not a Delta table (stores physical data).

In [ ]:
# CREATE a persistent view — stored in metastore, survives session end
spark.sql("""
    CREATE OR REPLACE VIEW day8_demo.long_haul_flights AS
    SELECT
        flight_date,
        origin,
        destination,
        distance_miles,
        delay_minutes
    FROM day8_demo.flights_delta
    WHERE distance_miles > 1500
""")

# Queryable just like a table
spark.sql("SELECT * FROM day8_demo.long_haul_flights ORDER BY distance_miles DESC LIMIT 5").show()

# Confirm it is a VIEW, not a TABLE
spark.sql("SHOW VIEWS IN day8_demo").show()

## 6. spark.table() — Loading a Registered Table as a DataFrame

`spark.table("db.table")` loads any registered table or view and returns a **DataFrame**.

```python
df = spark.table("database.table_name")
df.filter(...).groupBy(...).show()
```

| | `spark.table()` | `spark.sql()` |
|---|---|---|
| Argument | Table/view **name** | Full **SQL query string** |
| Returns | DataFrame | DataFrame |
| Use when | Loading a whole table | Running any SQL |

> **Exam trap:** `spark.table("SELECT * FROM sales")` is wrong — table name only, no SQL.
> `spark.sql("sales")` is wrong — sql() needs a full query string.

In [ ]:
# Load a registered table as a DataFrame
flights_df = spark.table("day8_demo.flights_delta")
flights_df.printSchema()

# Load a persistent view — same syntax
long_haul_df = spark.table("day8_demo.long_haul_flights")
long_haul_df.show(5)

# Chain any DataFrame operation on the result
from pyspark.sql.functions import col, avg

spark.table("day8_demo.flights_delta") \
     .groupBy("origin") \
     .agg(avg("delay_minutes").alias("avg_delay")) \
     .orderBy(col("avg_delay").desc()) \
     .show(5)

## 7. Delta Lake DML

Delta tables support full SQL DML. This is a key differentiator over plain Parquet.

| Command | Effect | Exam keyword |
|---|---|---|
| `INSERT INTO` | **Appends** rows — existing data untouched | "add without removing" |
| `INSERT OVERWRITE` | **Replaces all** rows in the table | "replace entire contents" |
| `UPDATE` | Modifies rows matching WHERE | "change values in existing rows" |
| `DELETE` | Removes rows matching WHERE | "remove rows" |
| `MERGE INTO` | Upsert — insert new, update existing | "sync source to target" |

> **Most tested:** `INSERT INTO` (append) vs `INSERT OVERWRITE` (full replace).

In [ ]:
# Create a small Delta table for DML examples
spark.sql("""
    CREATE OR REPLACE TABLE day8_demo.transactions (
        id      BIGINT,
        country STRING,
        amount  DOUBLE,
        status  STRING
    ) USING DELTA
""")

# ── INSERT INTO — appends rows ────────────────────────────────────────────────
spark.sql("""
    INSERT INTO day8_demo.transactions VALUES
    (1, 'NL', 1500.00, 'cleared'),
    (2, 'DE', 2300.00, 'cleared'),
    (3, 'RO',  900.00, 'pending')
""")

spark.sql("SELECT * FROM day8_demo.transactions").show()

In [ ]:
# INSERT INTO again — appends, does NOT replace existing rows
spark.sql("""
    INSERT INTO day8_demo.transactions VALUES
    (4, 'UK', 3100.00, 'cleared'),
    (5, 'FR',  450.00, 'failed')
""")

# 5 rows total — all previous rows are still there
spark.sql("SELECT COUNT(*) AS total FROM day8_demo.transactions").show()

In [ ]:
# INSERT OVERWRITE — replaces ALL existing data
spark.sql("""
    INSERT OVERWRITE day8_demo.transactions VALUES
    (10, 'NL', 9999.00, 'cleared')
""")

# Only 1 row remains — all previous data is gone
spark.sql("SELECT * FROM day8_demo.transactions").show()

# Restore for UPDATE / DELETE / MERGE demo
spark.sql("""
    INSERT INTO day8_demo.transactions VALUES
    (1, 'NL', 1500.00, 'cleared'),
    (2, 'DE', 2300.00, 'cleared'),
    (3, 'RO',  900.00, 'pending'),
    (4, 'UK', 3100.00, 'cleared'),
    (5, 'FR',  450.00, 'failed')
""")

In [ ]:
# UPDATE — modify rows matching a condition
spark.sql("""
    UPDATE day8_demo.transactions
    SET status = 'reviewed'
    WHERE status = 'pending'
""")

# DELETE — remove rows matching a condition
spark.sql("""
    DELETE FROM day8_demo.transactions
    WHERE status = 'failed'
""")

spark.sql("SELECT * FROM day8_demo.transactions ORDER BY id").show()

In [ ]:
# MERGE INTO — upsert: update existing rows, insert new ones in one pass
# Scenario: incoming batch has updates to existing rows and brand-new rows

from pyspark.sql import Row

updates = spark.createDataFrame([
    Row(id=2,  country='DE', amount=2300.00, status='reviewed'),  # existing — update
    Row(id=3,  country='RO', amount=900.00,  status='cleared'),   # existing — update
    Row(id=99, country='ES', amount=5500.00, status='cleared'),   # new — insert
])
updates.createOrReplaceTempView("updates_source")

spark.sql("""
    MERGE INTO day8_demo.transactions AS target
    USING updates_source AS source
    ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET target.status = source.status,
                   target.amount = source.amount
    WHEN NOT MATCHED THEN
        INSERT (id, country, amount, status)
        VALUES (source.id, source.country, source.amount, source.status)
""")

spark.sql("SELECT * FROM day8_demo.transactions ORDER BY id").show()

In [ ]:
# Cleanup — drop everything created today
spark.sql("DROP DATABASE IF EXISTS day8_demo CASCADE")
print("Cleanup complete.")

---

## Day 8 Checklist

- [ ] Know the metastore hierarchy: Catalog → Database → Table / View
- [ ] **Managed table:** omit LOCATION → DROP deletes data files
- [ ] **External table:** include LOCATION → DROP keeps data files
- [ ] `USING <format>` specifies format in CREATE TABLE (not FROM, not FORMAT)
- [ ] `OPTIONS (header='true', delimiter='|')` for CSV tables
- [ ] CTAS: `CREATE OR REPLACE TABLE ... AS SELECT ...` — schema from query, managed by default
- [ ] `CREATE VIEW` = persistent, metastore-backed, no physical data, cross-session
- [ ] `spark.table("db.table")` loads any registered table or view as a DataFrame
- [ ] `INSERT INTO` appends; `INSERT OVERWRITE` replaces all
- [ ] `UPDATE` / `DELETE` / `MERGE INTO` require Delta format

**Week 1 Complete — Certification deck now fully covered by course material.**

**Week 2:** Aggregations, Window functions, Joins, UDFs, Caching, Auto Loader